### Phase-I

#### Importing required libraries

In [8]:
# PDF Loading
from langchain_community.document_loaders import PyPDFLoader

# Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# Gemini
import google.generativeai as genai

# Environment Variables
from dotenv import load_dotenv
import os

# Misc
import numpy as np

#### Load Environment Variables

In [9]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
print("API Loaded Successfully")

API Loaded Successfully


#### Configure Gemini

In [10]:
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini Connected")

Gemini Connected


#### Load PDF

In [11]:
loader = PyPDFLoader(r"F:\GenAI\datasets\1.Introduction to Deep Learning.pdf")
documents = loader.load()
print(f"Pages Loaded: {len(documents)}")
 

Pages Loaded: 6


#### Inspect Data

In [12]:
print(documents[0].page_content[:1000])

What is Deep Learning? 
Deep learning is a subfield of AI and Machine Learning that is inspired by the structure of a human 
brain. 
Deep Learning does the same work as machine learning — that is, finding relationships between 
input and output and making predictions from data — but it goes a step further. While machine 
learning relies on statistical techniques to identify these relationships, deep learning uses a 
predefined logical structure called neural networks to automatically learn complex patterns from 
data.


#### Chunking

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 23


#### Check Chunk

In [14]:
print(chunks[0].page_content)

What is Deep Learning? 
Deep learning is a subfield of AI and Machine Learning that is inspired by the structure of a human 
brain. 
Deep Learning does the same work as machine learning — that is, finding relationships between 
input and output and making predictions from data — but it goes a step further. While machine 
learning relies on statistical techniques to identify these relationships, deep learning uses a


#### Embedding Model

In [15]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\DELL\AppData\Local\Temp\ipykernel_5200\412152783.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Create FAISS Vector Store

In [16]:
vector_store = FAISS.from_documents(chunks,embedding_model)

#### Save FAISS

In [17]:
vector_store.save_local("faiss_index")

### Phase-II

#### Load FAISS

In [18]:
vector_store = FAISS.load_local("faiss_index",embedding_model,allow_dangerous_deserialization=True)

#### User Query

In [19]:
query = "What is CNN?"

#### Retrieve Top Chunks

In [20]:
docs = vector_store.similarity_search(query,k=3)

#### Create Context

In [22]:
context = "\n\n".join([doc.page_content for doc in docs])
print(context)

pattern in the data. Output Layer which provides the final output. 
MLPs are versatile and can be used on various applications such as image recognition, speech 
recognition. 
 
2.Convolutional Neural Network (CNN) 
It is a type of neural network specifically designed for processing images. It is highly effective in tasks 
such as image recognition, object detection and more. 
 
3.Recurrent Neural Network (RNN)

Neural Networks (CNNs) are effective for image-related tasks, while Recurrent Neural Networks 
(RNNs) or Long Short-Term Memory (LSTM) networks are used for sequence data such as text or time 
series. 
 
5. Model Training 
During training, the model receives input data and produces predictions through forward 
propagation. The difference between the predicted and actual output is calculated using a loss

such as image recognition, object detection and more. 
 
3.Recurrent Neural Network (RNN) 
It is a type of neural network designed to handle sequential data like text, time ser

#### Build Prompt

In [23]:
prompt = f"""
You are an AI tutor.

Answer ONLY from the provided context.

Context:
{context}

Question:
{query}
"""

#### Send to Gemini

In [24]:
response = model.generate_content(prompt)
print(response.text)

Convolutional Neural Network (CNN) is a type of neural network specifically designed for processing images. It is highly effective in tasks such as image recognition, object detection, and more.
